In [7]:

%pip install -q requests
%pip install -q ratelimit
%pip install requests beautifulsoup4


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Ok so new problem new API calls
need to still import requests
db is now snp
example esearch: https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=snp&term=(TP53[Gene%20Name])%20AND%20%22pathogenic%22[Clinical%20Significance]%20
note partial encoding, hopefully old cold is still robust enough to handle any new exceptions

has a denotation fort clinical signif : pathogenic"[Clinical Significance]
above works as a boolean arg

aboves call RSIDs (ref SNP ID?)
so we take those, store em at least temporarily: might include in final matrix for future use.

IDs are called one at a time using this url: https://api.ncbi.nlm.nih.gov/variation/v0/refsnp/{ID}/frequency
Just gives demographics which is all we're interested in
some might return blank, especially if an underrepresented snp, as they have no ALFA data
see if we can find some way to filter for only samples which have ALFA data
could do it as an OR series, maybe

each group is denoted on the page as belonging to a specific group, list can be found here:https://www.ncbi.nlm.nih.gov/snp/docs/gsr/alfa/ALFA_20230706150541/ and here:https://www.ncbi.nlm.nih.gov/snp/docs/gsr/data_inclusion/#population 
Only 82 studies: if results are promising, we can try to check if studies include additional metadata, and individually tally counts for something like sex

but other than that; just need to format output: 
maybe for rows we do genes, columns are the SNP(A,C,G,T)(do we need U?N?) for each Ethnic Sample group, with counts for each. Maybe include an rsid as an additional column
seems big but im not sure of a good way to collapse any further without losing robustness for reading in SNPs

Some other things to note:
they track global minor allele frequency GMAF as an additional modifier. could be interesting to look at if the GMAF changes when reweighting
Also have [VALI] or validation status for whether or not certain metadata is present, ie by alfa[VALI] would only return results with validation status

add 2/24:
above likely too much. Have some better ideas

we winnow down to certain CLIN Categories (for now pathogenic and likely pathogenic, but may expand to include more, such as Conflicting-Interpretations-Of-Pathogenicity); reduces 900m to 183730 
 filtering further by ALFA va
 much more reasonable to work with
we can pull the explicit proportions 
we pull ALL The IDs for clinically significant SNPs( will need to do stepwise with retmax) with ALFA validation
we then do two searches for each ID; 
one search by ALFA frequency( https://api.ncbi.nlm.nih.gov/variation/v0/refsnp/{ID}/frequency) to get counts for each for each SNP

and one to get name/position of each SNP([What url gives name from RSID?](https://api.ncbi.nlm.nih.gov/variation/v0/refsnp/712) its seems its in there, just has a big response)
named should be under "locus". Maybe we simply store the RSID as rows, store gene name additionally as a column so we can collapse by gene easily. then we include ref as another column, and the counts for each nucleotide as the next columns
maybe subset by race, that way we have separate outputs for each race group

add 2/26:
we have main list of IDs: https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=snp&term=(by%20alfa[Validation]%20AND%20(pathogenic[Clinical_Significance]%20OR%20likely%20pathogenic[Clinical_Significance]))&retmax=1000

this should get us all significant SNPS with ALFA data: ~63k
there are some 12k snps with by frequency data and no ALFA: unsure if these are collapsable with ALFA
no ALFA SNPS without frequency; kinda obvious
significant SNPs with no frequency data: 107742 with no ALFA: ~120k
5.7k with pathogenic-likely-pathogenic(?what does this mean? some docu here: https://www.ncbi.nlm.nih.gov/clinvar/docs/clinsig/#:~:text=Prior%20to%202024%2C%20ClinVar%20had%20a%20single,somatic%20classification%20of%20clinical%20impact%2C%20and%20oncogenicity.) with freq data, figure out what this means
might want to expand criteria a bit for more SNPs, but 63k sounds good for now


final output should be a fairly large dataframe with SNP IDs as rows, and GENE, Ref allele, counts, and maybe positions as an additional list string 
some tutorials here
https://github.com/ncbi/dbsnp/blob/master/tutorials/Variation%20Services/Jupyter_Notebook
https://www.ncbi.nlm.nih.gov/snp/docs/entrez_help/

Things to discuss with Ed:
1. any good sources or locations to get reliable worldwide population data. It looks like ALFA is generally corresponding to worldwide samples. just using US census data might not be correct
could try and limit to SNPs within US population or only documented in US studies? dont really like this but can discuss with Ed
So far I found:
United Nations Statistics Division: http://data.un.org/Data.aspx?d=POP&f=tableCode:26
above seems to be reliable data, but will likely require manual curation and group collapse in order to properly categorize. Will likely need to cross reference with ALFA documentation
in order to ensure that groups are collapsed into proper categories. only issue is that only 109 countries are represented

one interesting facet of this is the opportunnity to look into which ethnicities fit into which categories along genomic lines rather than ethnic, cultural or geographic. IE turkish peoples are gmay generally be considered asian but in reality could be more similar geneticallly to eurpoeans or africans. A large lit search may be necessary

World Bank:https://data.worldbank.org/indicator/SP.POP.TOTL
seems to be accurate enough, but does not capture genetic discretion particularly well ie, displaced or minority ethnic groups exist within various nations (especially western 1st world) which are counted as homogenous within census data. could be useful for estimates, but not capturing depth we want

US Census: https://www.census.gov/data-tools/demo/idb/
contains international data, but like world bank does not discriminate along ethnic/genetic lines. has little to no ethnic data, in fact. could divide based on largest ethnic group by country, but thats messy and not as useful as WB or UNSD data

Our World in data: https://ourworldindata.org/population-growth
similar issue to US Census, little ethnographic data. Still, consider for supplementary info and potential for estimation

so what we can do to estimate is: find proportions of each ethnic group in UN database, then factor by most accurate estimate for total population (likely average of other 3 sources) for estimated worldwide proportions for each group. tricky but ultimately very doable 


NB: find data permission protocols for above databases, see what conditions and restrictions are for use, if any

2. Figure out what data to actually pull: preferably need some way to focus in on 922mil snps, but still try to capture reality of mutational landscape
so far, in my mind, limiting to clinical significance both a) reduces to the overall amount of snps for faster analysis and b) captures well documented snps with reliable data and known characterization, but having eds opinion would help as this kind of thing is more in his wheel house

3. Figuring out what groups need to be collapsed together
one interesting facet of this is the opportunnity to look into which ethnicities fit into which categories along genomic lines rather than ethnic, cultural or geographic. IE turkish peoples are gmay generally be considered asian but in reality could be more similar geneticallly to eurpoeans or africans. A large lit search may be necessary

4. if we want to look at all SNPs, specific AA change SNPs, or whether we care about every SNP or just the main variant from reference allele

ADD 3/7
alright so ed wants the following things:
"One first plot that would be very interesting:

x-axis: the overall total frequency from ALFA (i.e., the unweighted percentage)

y-axis: the weighted average using these #s and the individual ALFA frequencies

Plot: Scatter plot; each point is the (ALFA reported “total” frequency = x; weighted average = y)


That scatter plot will show us how big the corrections are… if it’s the “x=y” line, there is no difference.  We’ll see how much it deviates from x=y and if anything jumps out re: trends.
Also interesting:

Can you make a waterfall type plot, where you:

    Find the difference in (Weighted Average) – (ALFA reported “total” frequency) for each SNP
    Put then in order from biggest to smallest
    Then, you plot the scatter plot where x = rank (from 1 to max) and y = the difference found in 1).

This will be a way to get a sense of what proportion of all of the snps considered had very large, large, modest, no changes… and also show over and under estimates from the naïve/ALFA totals vs the weighted changes.

Lastly – if you can make an Excel table that lists all the SNPs (SNP ID, gene name, status (pathogenic, etc.), all the frequencies from ALFA (total as well as each subcategory), and the weighted average… that would be helpful for me to look at and “mine” and see if there are any interesting things to dig into deeper."


So we need to address all of that.

Also: new terms for clinSig to consider: drug-response, protective, risk factor

In [1]:
from requests import get, codes as http_code
import pandas as pd
import json
import time
import requests
import os
from datetime import datetime
import openpyxl
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
import re
from IPython.display import clear_output

In [28]:
reply=get("https://api.ncbi.nlm.nih.gov/variation/v0/refsnp/712/frequency".format(16))
reply.json()

{'build_id': '20231103111315',
 'results': {'1@25209617': {'ref': 'A',
   'counts': {'PRJNA507278': {'allele_counts': {'SAMN10492705': {'C': 10387,
       'A': 10231},
      'SAMN10492695': {'C': 8807, 'A': 7207},
      'SAMN10492703': {'C': 707, 'A': 2239},
      'SAMN10492696': {'C': 21, 'A': 93},
      'SAMN10492698': {'C': 686, 'A': 2146},
      'SAMN10492704': {'C': 85, 'A': 27},
      'SAMN10492697': {'C': 61, 'A': 25},
      'SAMN10492701': {'C': 24, 'A': 2},
      'SAMN10492699': {'C': 67, 'A': 79},
      'SAMN10492700': {'C': 329, 'A': 281},
      'SAMN10492702': {'C': 60, 'A': 38},
      'SAMN11605645': {'C': 332, 'A': 360}}}}}}}

In [2]:

#FIXME add an error list reader

# Define the base URL
base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
query = "(by%20alfa[Validation]%20AND%20(pathogenic[Clinical_Significance]%20OR%20likely%20pathogenic[Clinical_Significance]))"
retmax = 1000  # Number of SNPs retrieved per request
retstart=0
print(f"{base_url}?db=snp&term={query}&retmax=1")
# Directory for storing raw data outputs
cwd_Raw_Data_outputs = os.path.join(os.getcwd(), "Raw_Data_outputs")
os.makedirs(cwd_Raw_Data_outputs, exist_ok=True)  # Ensure directory exists

# Get the total number of SNPs
response = requests.get(f"{base_url}?db=snp&term={query}&retmax=1")
if response.status_code == 200:
    root = ET.fromstring(response.text)
    count_element = root.find("Count")
    
    if count_element is not None:
        SNPcount = int(count_element.text)  # Total SNP count
        print(f"Total SNP Count: {SNPcount}")
    else:
        print("Error: <Count> element not found in response.")
        SNPcount = 0
else:
    print(f"Error: {response.status_code}")
    SNPcount = 0

# Retrieve SNP IDs in batches of 1000
snp_ids = []
num_batches = (SNPcount // retmax) + 1  # Number of searches needed
for i in range(num_batches):
    retstart = i * retmax  # Offset for pagination
    prefix = "RawData_"+str(retstart)+"_"  # The prefix you're looking for
    print(f"searching for {prefix}")
    # Find the first matching file
    matching_file = next((file for file in os.listdir(cwd_Raw_Data_outputs) if file.startswith(prefix)), None)
    if matching_file:
        print(f"Matching file found: {matching_file}")
        rawFreeze=os.path.join(cwd_Raw_Data_outputs, matching_file)
        with open(rawFreeze, "r", encoding="utf-8") as file:
            response_text = file.read()
            batch_ids = re.findall(r"<Id>(\d+)</Id>", response_text)
            # Convert extracted IDs to a list of strings
            snp_ids.extend(batch_ids)
            print(f"Batch {i+1}: Retrieved {len(batch_ids)} SNPs. Total so far: {len(snp_ids)}")

    else:
        print("raw data not found. making query...")
        paginated_url = f"{base_url}?db=snp&term={query}&retmax={retmax}&retstart={retstart}"
        
        response = requests.get(paginated_url)
        if response.status_code == 200:
            root = ET.fromstring(response.text)
            batch_ids = [id_elem.text for id_elem in root.findall(".//IdList/Id")]
            
            # Append new IDs to the list
            snp_ids.extend(batch_ids)
            print(f"Batch {i+1}: Retrieved {len(batch_ids)} SNPs. Total so far: {len(snp_ids)}")
            
            # Save raw response to a text file
            timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
            filename = os.path.join(cwd_Raw_Data_outputs, f"RawData_{retstart}_{timestamp}.txt")
            
            with open(filename, "w", encoding="utf-8") as file:
                file.write(f"Search Date & Time: {timestamp}\n")
                file.write(f"Request URL: {paginated_url}\n")
                file.write(f"Response Data:\n{response.text}\n")
            
            print(f"Saved raw response to: {filename}")

            # Stop if we've reached the full count
            if len(snp_ids) >= SNPcount:
                break
            time.sleep(1)# **Rate limiting**: Sleep for 1 second to comply with NCBI's API restrictions

        else:
            print(f"Error retrieving batch {i+1}: {response.status_code}")
            time.sleep(1)
            break  # Stop on error
# Final check
print(f"Final SNP count retrieved: {len(snp_ids)}")

output_file = os.path.join(os.getcwd(), "snp_ids.json")
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(snp_ids, f, indent=4)
print(f"SNP IDs saved to {output_file}")

https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=snp&term=(by%20alfa[Validation]%20AND%20(pathogenic[Clinical_Significance]%20OR%20likely%20pathogenic[Clinical_Significance]))&retmax=1
Total SNP Count: 82470
searching for RawData_0_
Matching file found: RawData_0_2025-03-13_13-24-11.txt
Batch 1: Retrieved 1000 SNPs. Total so far: 1000
searching for RawData_1000_
Matching file found: RawData_1000_2025-03-13_13-24-13.txt
Batch 2: Retrieved 1000 SNPs. Total so far: 2000
searching for RawData_2000_
Matching file found: RawData_2000_2025-03-13_13-24-15.txt
Batch 3: Retrieved 1000 SNPs. Total so far: 3000
searching for RawData_3000_
Matching file found: RawData_3000_2025-03-13_13-24-17.txt
Batch 4: Retrieved 1000 SNPs. Total so far: 4000
searching for RawData_4000_
Matching file found: RawData_4000_2025-03-13_13-24-18.txt
Batch 5: Retrieved 1000 SNPs. Total so far: 5000
searching for RawData_5000_
Matching file found: RawData_5000_2025-03-13_13-24-21.txt
Batch 6: Retrieved 1000 

KeyboardInterrupt: 

#rj = reply.json()
#rj['results']['1@11563271']['counts']['PRJNA507278']['allele_counts']['SAMN10492695']['C']#Notice JSON Structure here; notation can be used to limit freeze sizes


def get_snp_info(snp_id):
    """
    Queries NCBI Variation API for clinical significance and gene name (locus) of a given SNP ID.
    """
    url = f"https://api.ncbi.nlm.nih.gov/variation/v0/refsnp/{snp_id}"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        
        # Initialize default values
        clinical_significance = "N/A"
        gene_name = "N/A"

        # Extract clinical significance if available
        if "clinical_significances" in data:
            clinical_significance = data["clinical_significances"]

        # Extract gene name (locus)
        try:
            gene_name = data["primary_snapshot_data"]["allele_annotations"][0] \
                        ["assembly_annotation"][0]["genes"][0]["locus"]
        except (KeyError, IndexError, TypeError):
            gene_name = "N/A"  # If not found, return N/A
        
        return {
            "snp_id": snp_id,
            "gene_name": gene_name,
            "clinical_significance": clinical_significance
        }
    
    else:
        print(f"Error retrieving SNP {snp_id}: HTTP {response.status_code}")
        return {"snp_id": snp_id, "gene_name": "N/A", "clinical_significance": "N/A"}

# Example usage
snp_id = 671 # Example SNP ID
snp_info = get_snp_info(snp_id)
print(json.dumps(snp_info, indent=4))

In [3]:


from requests import get, codes as http_code
from ratelimit import limits
from typing import Any
# Directory for storing raw JSON responses
cwd_Raw_Data_outputs = os.path.join(os.getcwd(), "Raw_Data_outputs")
os.makedirs(cwd_Raw_Data_outputs, exist_ok=True)  # Ensure directory exists

# Define error log file
error_log_file = os.path.join(cwd_Raw_Data_outputs, "error_log.txt")

#@limits(calls=1, period=1)  # Only one call per second
def get_frequency_for(rs_id: str) -> Any:
    """
    Retrieve frequency data by rsid in JSON format.
    Checks for an existing freeze before querying the API.
    If a freeze exists, it is read instead of making a new API request.
    Logs errors instead of raising exceptions.
    """
    # Check if a previous freeze exists
    existing_files = [file for file in os.listdir(cwd_Raw_Data_outputs) if file.startswith(f"RawData_Frequency_{rs_id}_")]

    if existing_files:
        # Use the latest freeze file
        latest_file = max(existing_files, key=lambda f: os.path.getctime(os.path.join(cwd_Raw_Data_outputs, f)))
        file_path = os.path.join(cwd_Raw_Data_outputs, latest_file)
        
        #print(f"Using stored frequency freeze: {file_path}")
        
        # Load data from the existing file
        with open(file_path, "r", encoding="utf-8") as file:
            return json.load(file)

    # If no freeze exists, query the API
    BYRSID_URL = f"https://api.ncbi.nlm.nih.gov/variation/v0/refsnp/{rs_id}/frequency"
    
    reply = get(BYRSID_URL)
    
    if reply.status_code != http_code.ok:
        error_message = f"Request failed: HTTP {reply.status_code} for RSID {rs_id}\n"
        print(error_message)
        with open(error_log_file, "a", encoding="utf-8") as log_file:
            log_file.write(error_message)
        return []  # Return empty list instead of raising an exception

    content_type = reply.headers.get('content-type', '')
    if content_type != 'application/json':
        error_message = f"Unexpected content type: {content_type} for RSID {rs_id}\n"
        print(error_message)
        with open(error_log_file, "a", encoding="utf-8") as log_file:
            log_file.write(error_message)
        return []  # Return empty list instead of raising an exception

    # Parse JSON response
    data = reply.json()

    # Save the raw JSON response with the RSID and timestamp
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = os.path.join(cwd_Raw_Data_outputs, f"RawData_Frequency_{rs_id}_{timestamp}.json")

    with open(filename, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=4)

    print(f"Saved frequency data to: {filename}")

    return data



In [4]:


# Define the SNP IDs file path
snp_ids_file = os.path.join(os.getcwd(), "snp_ids.json")

# Directory where frequency freezes are stored
cwd_Raw_Data_outputs = os.path.join(os.getcwd(), "Raw_Data_outputs")
error_log_file = os.path.join(os.getcwd(), "Raw_Data_outputs", "error_log.txt")
os.makedirs(cwd_Raw_Data_outputs, exist_ok=True)  # Ensure directory exists

# Load SNP IDs from JSON file
if os.path.exists(snp_ids_file):
    with open(snp_ids_file, "r", encoding="utf-8") as file:
        snp_ids = json.load(file)
    print(f"Loaded {len(snp_ids)} SNP IDs from {snp_ids_file}")
else:
    print("SNP IDs JSON file not found.")
    snp_ids = []

# Read error_log.txt and extract failed RSIDs
failed_ids = set()  # Use a set to avoid duplicates
if os.path.exists(error_log_file):
    with open(error_log_file, "r", encoding="utf-8") as file:
        for line in file:
            match = re.search(r"for RSID (\d+)", line)  # Extract RSID from error log
            if match:
                failed_ids.add(match.group(1))  # Store as string to match snp_ids format

# Remove failed RSIDs from the SNP list
filtered_snp_ids = [rsid for rsid in snp_ids if rsid not in failed_ids]
with open(snp_ids_file, "w", encoding="utf-8") as file:
     json.dump(filtered_snp_ids, file, indent=4)

# Print results
print(f"Removed {len(snp_ids) - len(filtered_snp_ids)} failed SNPs from the list.")
print(f"Final SNP count: {len(filtered_snp_ids)}")

# Fetch frequency data for each RSID, checking for stored freeze first
for rsid in snp_ids:
    print(f"\nFetching data for RSID: {rsid}")

    # Check if a previous freeze exists
    existing_files = [file for file in os.listdir(cwd_Raw_Data_outputs) if file.startswith(f"RawData_Frequency_{rsid}_")]

    if existing_files:
        # Use the latest freeze file
        latest_file = max(existing_files, key=lambda f: os.path.getctime(os.path.join(cwd_Raw_Data_outputs, f)))
        file_path = os.path.join(cwd_Raw_Data_outputs, latest_file)
        
        print(f"Using stored frequency freeze: {file_path}")
        
        # Load data from the existing file
        with open(file_path, "r", encoding="utf-8") as file:
            freq_data = json.load(file)
    else:
        # If no freeze exists, query the API
        time.sleep(1)
        freq_data = get_frequency_for(rsid)

    # Clear output for better visualization
    clear_output(wait=True)
    #print(json.dumps(freq_data, indent=4))  # Print formatted JSON output


Fetching data for RSID: 334
Saved frequency data to: c:\Users\david\dbSNP\Raw_Data_outputs\RawData_Frequency_334_2025-03-19_10-49-40.json


In [4]:

cwd_Raw_Data_outputs = os.path.join(os.getcwd(), "Raw_Data_outputs")
os.makedirs(cwd_Raw_Data_outputs, exist_ok=True)  # Ensure directory exists

def get_snp_info(snp_id):
    """
    Queries NCBI Variation API for clinical significance and gene name (locus) of a given SNP ID.
    If a stored JSON response exists, it is read instead of making a new API request.
    """
    # Check if a previous freeze exists
    existing_files = [file for file in os.listdir(cwd_Raw_Data_outputs) if file.startswith(f"RawData_SNP_{snp_id}_")]
    
    if existing_files:
        # Use the latest freeze file
        latest_file = max(existing_files, key=lambda f: os.path.getctime(os.path.join(cwd_Raw_Data_outputs, f)))
        file_path = os.path.join(cwd_Raw_Data_outputs, latest_file)
        
        #print(f"Using stored JSON freeze: {file_path}")
        
        # Load data from the existing file
        with open(file_path, "r", encoding="utf-8") as file:
            data = json.load(file)
    
    else:
        # No freeze found, query the API
        url = f"https://api.ncbi.nlm.nih.gov/variation/v0/refsnp/{snp_id}"
        response = requests.get(url)

        if response.status_code == 200:
            data = response.json()

            # Save the raw JSON response with the SNP ID and timestamp
            timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
            filename = os.path.join(cwd_Raw_Data_outputs, f"RawData_SNP_{snp_id}_{timestamp}.json")
            
            with open(filename, "w", encoding="utf-8") as file:
                json.dump(data, file, indent=4)
            
            print(f"Saved raw JSON response to: {filename}")
        
        else:
            print(f"Error retrieving SNP {snp_id}: HTTP {response.status_code}")
            return {"snp_id": snp_id, "gene_name": "N/A", "clinical_significance": ["N/A"]}

    # Initialize default values
    clinical_significances = []  # Store all valid clinical significance values
    gene_name = "N/A"

    # Extract gene name (locus)
    try:
        gene_name = data["primary_snapshot_data"]["allele_annotations"][0] \
                    ["assembly_annotation"][0]["genes"][0]["locus"]
    except (KeyError, IndexError, TypeError):
        gene_name = "N/A"  # If not found, return N/A
    
    # Extract all clinical significance values across SNP variants
    try:
        for variant in data["primary_snapshot_data"]["allele_annotations"]:
            clinical_data = variant.get("clinical", [])  # Get clinical data list
            
            if clinical_data:  # Check if clinical data exists
                for entry in clinical_data:
                    if "clinical_significances" in entry:
                        clinical_significances.extend(entry["clinical_significances"])
    
    except (KeyError, IndexError, TypeError):
        clinical_significances = ["N/A"]  # If not found, return "N/A"

    # Remove duplicates, if any
    clinical_significances = list(set(clinical_significances)) if clinical_significances else ["N/A"]

    return {
        "snp_id": snp_id,
        "gene_name": gene_name,
        "clinical_significance": clinical_significances
    }


In [5]:

# Directory for storing raw JSON responses
cwd_Raw_Data_outputs = os.path.join(os.getcwd(), "Raw_Data_outputs")
os.makedirs(cwd_Raw_Data_outputs, exist_ok=True)  # Ensure directory exists

# Define the file path for SNP IDs
snp_ids_file = os.path.join(os.getcwd(), "snp_ids.json")

# Check if the file exists and read it
if os.path.exists(snp_ids_file):
    with open(snp_ids_file, "r", encoding="utf-8") as file:
        snp_ids = json.load(file)
    print(f"Loaded {len(snp_ids)} SNP IDs from {snp_ids_file}")
else:
    print("SNP IDs JSON file not found.")
    snp_ids = []

# Initialize a list to store SNP info
snp_data = []

# Loop through all SNP IDs and retrieve info
for snp_id in snp_ids:
    print(f"Processing SNP ID: {snp_id}")

    # Get clinical significance and gene name
    snp_info = get_snp_info(snp_id)
    clear_output(wait=True)
    # Append extracted data to list
    snp_data.append({
        "SNP ID": snp_id,
        "Gene Name": snp_info["gene_name"],
        "Clinical Significance": ", ".join(snp_info["clinical_significance"]),
    })

print("\nSNP data retrieval complete!")

Processing SNP ID: 1838031568


KeyboardInterrupt: 

In [ ]:
# File and directory paths
cwd_Raw_Data_outputs = os.path.join(os.getcwd(), "Raw_Data_outputs")
os.makedirs(cwd_Raw_Data_outputs, exist_ok=True)

snp_ids_file = os.path.join(os.getcwd(), "snp_ids.json")
output_csv = os.path.join(os.getcwd(), "SNP_Full_Export.csv")

# Fixed population group IDs based on known structure
population_ids = [
    "SAMN10492705", "SAMN10492695", "SAMN10492703", "SAMN10492696",
    "SAMN10492698", "SAMN10492704", "SAMN10492697", "SAMN10492701",
    "SAMN10492699", "SAMN10492700", "SAMN10492702", "SAMN11605645"
]

# Load SNP IDs
if os.path.exists(snp_ids_file):
    with open(snp_ids_file, "r", encoding="utf-8") as f:
        snp_ids = json.load(f)
    print(f"Loaded {len(snp_ids)} SNP IDs.")
else:
    print("SNP IDs JSON file not found.")
    snp_ids = []

# Prepare list to store data
results = []

# Process each SNP
for snp_id in snp_ids:
    print(f"Processing SNP ID: {snp_id}")

    # Define the path for the freeze file
    freeze_filename = os.path.join(cwd_Raw_Data_outputs, f"RowData_{snp_id}.json")

    # Check if the freeze file already exists
    if os.path.exists(freeze_filename):
        print(f"Found row freeze for {snp_id}, loading from file...")
        with open(freeze_filename, "r", encoding="utf-8") as f:
            row = json.load(f)
        results.append(row)
        continue

    # Get SNP info
    snp_info = get_snp_info(snp_id)
    
    # Get frequency info
    freq_data = get_frequency_for(snp_id)
    if not isinstance(freq_data, dict):
        continue

    # Extract reference allele and population data
    try:
        results_data = freq_data["results"]
        pos_key = next(iter(results_data))  # Get dynamic position key
        ref_allele = results_data[pos_key]["ref"]
        allele_counts = results_data[pos_key]["counts"]["PRJNA507278"]["allele_counts"]
    except (KeyError, StopIteration, TypeError):
        print(f"Failed to parse frequency data for RSID {snp_id}")
        continue

    # Build row with population counts for reference allele
    row = {
        "SNP ID": snp_id,
        "Gene Name": snp_info.get("gene_name", "N/A"),
        "Clinical Significance": snp_info.get("clinical_significance", ["N/A"]),
        "Reference Allele": ref_allele
    }

    for pop_id in population_ids:
        count = allele_counts.get(pop_id, {}).get(ref_allele, 0)
        row[pop_id] = count

    # Save the row freeze
    with open(freeze_filename, "w", encoding="utf-8") as f:
        json.dump(row, f, indent=4)
    print(f"Saved row freeze for {snp_id} to {freeze_filename}")

    clear_output(wait=True)
    results.append(row)

# Create DataFrame
df = pd.DataFrame(results)
df.set_index("SNP ID", inplace=True)

# Save to CSV
df.to_csv(output_csv)
print(f"\nExport complete! CSV saved to: {output_csv}")

Loaded 79152 SNP IDs.
Processing SNP ID: 2542522575
Found row freeze for 2542522575, loading from file...
Processing SNP ID: 2536790447
Found row freeze for 2536790447, loading from file...
Processing SNP ID: 2527358629
Found row freeze for 2527358629, loading from file...
Processing SNP ID: 2524698132
Found row freeze for 2524698132, loading from file...
Processing SNP ID: 2521458985
Found row freeze for 2521458985, loading from file...
Processing SNP ID: 2518961833
Found row freeze for 2518961833, loading from file...
Processing SNP ID: 2517861714
Found row freeze for 2517861714, loading from file...
Processing SNP ID: 2513091106
Found row freeze for 2513091106, loading from file...
Processing SNP ID: 2506484397
Found row freeze for 2506484397, loading from file...
Processing SNP ID: 2505396805
Found row freeze for 2505396805, loading from file...
Processing SNP ID: 2503751017
Found row freeze for 2503751017, loading from file...
Processing SNP ID: 2494585590
Found row freeze for 249

KeyboardInterrupt: 

In [ ]:
#counts for people with euro ancestry
md_json=get("https://api.ncbi.nlm.nih.gov/variation/v0/metadata/frequency").json()
md = {}
for project_json in md_json:
  p = {}
  p['json']=project_json
  p['pops']={}
  md[project_json['bioproject_id']] = p

def add_all_pops(populations, project):
  for p in populations:
    project['pops'][p['biosample_id']] = p
  if 'subs' in p:
    add_all_pops(p['subs'], project)

for prj_id, prj in md.items():
  add_all_pops(prj['json']['populations'], prj)

print(md['PRJNA507278']['json']['short_name'])
print(md['PRJNA507278']['pops']['SAMN10492695']['name'])

TypeError: get_frequency_for() got an unexpected keyword argument 'indent'

In [ ]:

def print_study_counts(study_id: str, study_counts: Any) -> None:
  """
  Print counts per study

  At present, we only offer counts per allele,
  not yet per genotype
  """
  print("\tAllele counts for study: {}".format(study_id))
  allele_counts = study_counts["allele_counts"]

  for pop_id, pop_counts in allele_counts.items():
    print("\t\tAllele counts for population {}".format(pop_id))
    for allele, count in pop_counts.items():
      print("\t\t\tAllele: {}. Count: {}".format(
        allele, count))

In [ ]:
frequency_data = get_frequency_for(rs_id=16)
for interval, freq_by_pop in frequency_data["results"].items():
  # Each key describes an interval
  # in <length>@<start> format
  length, start = interval.split("@")
  print("Start: {}. Length: {}. Ref. Allele: {}".format(
    start, length, freq_by_pop["ref"]))
  counts_per_study = freq_by_pop["counts"]

  # Print counts per study
  for study_id, study_counts in counts_per_study.items():
    print_study_counts(study_id, study_counts)